# Unidad 9 · ANOVA de dos factores, interacción y supuestos

Cuaderno de la base abierta de ingeniería (Análisis de Datos). Reproduce el ejemplo resuelto (impresión 3D:
material × orientación) y la pieza del caso Estampa Sur (secado según tinta y temperatura). Los datos están en
`../datos/`: `u09_ejemplo.csv` lo genera el verificador de la unidad y `secado.csv` el script del caso.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

## 1. Ejemplo · resistencia de probetas impresas en 3D

Primero la tabla ANOVA de dos factores con interacción. En un diseño balanceado, las sumas de cuadrados
tipo I y tipo II son iguales.

In [ ]:
ej = pd.read_csv("../datos/u09_ejemplo.csv")
print(ej.groupby(["material", "orientacion"]).resistencia.agg(["mean", "var", "count"]))
mod_ej = smf.ols("resistencia ~ C(material) * C(orientacion)", data=ej).fit()
tabla_ejemplo = sm.stats.anova_lm(mod_ej, typ=2)
print(tabla_ejemplo)

## 2. La forma matricial $Y = X\beta + \varepsilon$

Codificación de referencia: PLA y horizontal son la base. Las columnas son la constante, «es PETG»,
«es vertical» y su producto (la interacción).

In [ ]:
X = np.column_stack([np.ones(len(ej)),
                     (ej.material == "PETG").astype(float),
                     (ej.orientacion == "vertical").astype(float),
                     ((ej.material == "PETG") & (ej.orientacion == "vertical")).astype(float)])
y = ej.resistencia.to_numpy(float)
XtX, Xty = X.T @ X, X.T @ y
beta_ejemplo = np.linalg.solve(XtX, Xty)
print("X'X =\n", XtX)
print("X'y =", Xty)
print("beta =", beta_ejemplo)   # [media PLA-horizontal, efecto PETG, efecto vertical, interacción]

In [ ]:
medias = ej.groupby(["orientacion", "material"]).resistencia.mean().unstack()
medias.plot(marker="o", title="Gráfico de interacción (ejemplo)")
plt.ylabel("resistencia media (MPa)")
plt.show()
print("Levene:", stats.levene(*[g.resistencia for _, g in ej.groupby(["material", "orientacion"])]))
print("Shapiro de los residuos:", stats.shapiro(mod_ej.resid))

## 3. Caso Estampa Sur · secado según tinta y temperatura

Tres tintas × dos temperaturas × cinco réplicas. Se mira primero la interacción; si no es significativa,
se leen los efectos principales.

In [ ]:
s = pd.read_csv("../datos/secado.csv")
mod_caso = smf.ols("secado_min ~ C(tinta) * C(temperatura)", data=s).fit()
tabla_caso = sm.stats.anova_lm(mod_caso, typ=2)
print(tabla_caso)
print(s.groupby(["tinta", "temperatura"]).secado_min.mean().unstack())

In [ ]:
levene_caso = stats.levene(*[g.secado_min for _, g in s.groupby(["tinta", "temperatura"])])
print("Levene (seis celdas):", levene_caso)
print("Levene (tres tintas, como el script del caso):", stats.levene(*[g.secado_min for _, g in s.groupby("tinta")]))
print("Shapiro de los residuos:", stats.shapiro(mod_caso.resid))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].scatter(mod_caso.fittedvalues, mod_caso.resid); ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_xlabel("ajustado"); ax[0].set_ylabel("residuo")
sm.qqplot(mod_caso.resid, line="s", ax=ax[1])
plt.tight_layout(); plt.show()

In [ ]:
# Un factor (unidad 7) frente a dos factores: la temperatura sale del error y la F de la tinta crece
print(sm.stats.anova_lm(smf.ols("secado_min ~ C(tinta)", data=s).fit(), typ=1))